# 01 · Dataset overview

This notebook audits the repository’s raw fake/real news files before modeling. The labels are assigned from the source filename (`Fake.csv` → `fake`, `True.csv` → `real`); that is the dataset’s labeling scheme, not a universal definition of fake news. This notebook does not train a model or make claims about performance on news outside this dataset.

## Problem framing checkpoint

- **Task:** binary classification of articles as fake or real using the supplied labels.
- **Primary metric for later model selection:** macro F1, so each class contributes equally to the score. Supplementary metrics can describe the error trade-offs.
- **Model family constraint:** Naive Bayes only, as specified by the project owner. The Naive Bayes variant, text representation (for example, counts or TF-IDF), and tuning choices remain undecided until the model-building stage.
- **Initial risks:** source-file labels may be reflected in `subject`, dates, or writing and formatting conventions; duplicates may inflate validation scores; performance on this dataset may not transfer to other publishers, topics, or time periods.

The primary metric and data assumptions are explicit here so later model comparisons can be judged against a declared criterion. Revisit this framing only if the audit below reveals a material issue with the task or labels.

## 1. Reproducible environment and data loading

The global seed is **42**. It is set for Python’s `random` and NumPy here; later modeling code should reuse `GLOBAL_SEED` for any estimator or split that accepts a `random_state`. Package versions are captured from the active notebook kernel so the recorded values correspond to the environment that runs this notebook.

In [1]:
from pathlib import Path
import importlib.metadata as package_metadata
import platform
import random

import numpy as np
import pandas as pd

GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

packages_to_record = [
    'pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn', 'nltk', 'ipykernel'
]
environment = {'python': platform.python_version()}
for package in packages_to_record:
    try:
        environment[package] = package_metadata.version(package)
    except package_metadata.PackageNotFoundError:
        environment[package] = 'not installed'

print('Global seed:', GLOBAL_SEED)
print('Environment versions:')
for package, version in environment.items():
    print(f'  {package}: {version}')

# Find the repository root from the current notebook working directory.
search_start = Path.cwd().resolve()
repository_candidates = [search_start, *search_start.parents]
PROJECT_ROOT = next(
    (candidate for candidate in repository_candidates
     if (candidate / 'data' / 'raw' / 'News').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Could not find data/raw/News from the notebook working directory. '
        'Open Jupyter from the repository or set PROJECT_ROOT explicitly.'
    )

RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'News'
source_files = {'fake': RAW_DATA_DIR / 'Fake.csv', 'real': RAW_DATA_DIR / 'True.csv'}
missing_files = [str(path) for path in source_files.values() if not path.is_file()]
if missing_files:
    raise FileNotFoundError(f'Required dataset file(s) not found: {missing_files}')

frames = []
for label, path in source_files.items():
    frame = pd.read_csv(path)
    frame['label'] = label
    frame['source_file'] = path.name
    frames.append(frame)

news = pd.concat(frames, ignore_index=True)
print('Repository root:', PROJECT_ROOT)
print('Loaded files:', {label: path.name for label, path in source_files.items()})
print('Combined shape:', news.shape)

Global seed: 42
Environment versions:
  python: 3.13.3
  pandas: 3.0.6
  numpy: 2.5.3
  scikit-learn: 1.9.1
  matplotlib: 3.11.2
  seaborn: 0.13.2
  nltk: 3.10.3
  ipykernel: 7.3.0
Repository root: C:\Projects_Tracking\News_Spam_Classifier
Loaded files: {'fake': 'Fake.csv', 'real': 'True.csv'}
Combined shape: (44898, 6)


## 2. Dataset size, schema, and resource footprint

Treat `title`, `text`, `subject`, and `date` as candidate input fields during this audit. `label` is the target derived from the filename; `source_file` is provenance metadata and must not be used as a model feature because it directly encodes the label. These distinctions will matter when building leakage-safe pipelines.

In [2]:
candidate_features = ['title', 'text', 'subject', 'date']
required_columns = set(candidate_features)
missing_columns = required_columns.difference(news.columns)
if missing_columns:
    raise ValueError(f'Expected dataset columns are missing: {sorted(missing_columns)}')

schema = pd.DataFrame({
    'dtype': news.dtypes.astype(str),
    'non_null_rows': news.notna().sum(),
    'missing_rows': news.isna().sum(),
    'missing_percent': (news.isna().mean() * 100).round(3),
})
print('Rows:', f'{len(news):,}')
print('Candidate feature count:', len(candidate_features))
print('Columns including target/provenance:', len(news.columns))
print('Combined DataFrame memory (MiB):', round(news.memory_usage(deep=True).sum() / 1024**2, 2))
display(schema)

Rows: 44,898
Candidate feature count: 4
Columns including target/provenance: 6
Combined DataFrame memory (MiB): 152.25


,dtype,non_null_rows,missing_rows,missing_percent
title,str,44898,0,0.0
text,str,44898,0,0.0
subject,str,44898,0,0.0
date,str,44898,0,0.0
label,str,44898,0,0.0
source_file,str,44898,0,0.0


## 3. Target distribution and source-file integrity

Duplicate rows are quantified before deciding whether any should be removed. Do not drop records at this stage: the right treatment depends on where duplicates occur and how the later validation split is designed. Exact duplicates across the article fields but with conflicting labels would be a particularly important label-quality concern.

In [3]:
class_summary = (
    news['label'].value_counts()
    .rename_axis('label')
    .to_frame('rows')
    .assign(percent=lambda table: (table['rows'] / len(news) * 100).round(2))
)
print('Class distribution:')
display(class_summary)

source_summary = []
for label, path in source_files.items():
    frame = news.loc[news['label'].eq(label), candidate_features]
    source_summary.append({
        'source_file': path.name,
        'label': label,
        'rows': len(frame),
        'exact_duplicate_rows_after_first': int(frame.duplicated().sum()),
        'duplicate_percent': round(frame.duplicated().mean() * 100, 3),
    })
display(pd.DataFrame(source_summary).set_index('source_file'))

article_duplicates = news.duplicated(subset=candidate_features, keep=False)
conflicting_duplicate_groups = (
    news.loc[article_duplicates]
    .groupby(candidate_features, dropna=False)['label']
    .nunique()
    .gt(1)
    .sum()
)
print('Rows participating in exact duplicate article groups:', int(article_duplicates.sum()))
print('Duplicate article groups with conflicting labels:', int(conflicting_duplicate_groups))

Class distribution:


,rows,percent
label,,
fake,23481,52.3
real,21417,47.7


,label,rows,exact_duplicate_rows_after_first,duplicate_percent
source_file,,,,
Fake.csv,fake,23481,3,0.013
True.csv,real,21417,206,0.962


Rows participating in exact duplicate article groups: 405
Duplicate article groups with conflicting labels: 0


## 4. Missingness, date coverage, and initial data risks

Dates are parsed for an initial coverage check only. Their format and distribution should be inspected before considering any temporal split. This dataset’s date field may reflect article publication, collection, or export conventions; a parsed date alone does not prove a trustworthy time ordering.

In [4]:
quality_rows = []
for column in candidate_features:
    missing = news[column].isna()
    blank = news[column].astype('string').str.strip().eq('').fillna(False)
    quality_rows.append({
        'column': column,
        'missing_values': int(missing.sum()),
        'blank_strings': int(blank.sum()),
        'unique_values': int(news[column].nunique(dropna=True)),
    })
display(pd.DataFrame(quality_rows).set_index('column'))

parsed_dates = pd.to_datetime(news['date'].astype('string').str.strip(), errors='coerce')
valid_date_count = int(parsed_dates.notna().sum())
print('Dates parsed:', f'{valid_date_count:,}', '/', f'{len(news):,}')
print('Unparseable dates:', int(parsed_dates.isna().sum()))
if valid_date_count:
    print('Date coverage:', parsed_dates.min().date(), 'to', parsed_dates.max().date())
print('Date parsing by label:')
display(pd.DataFrame({
    'label': news['label'],
    'date_parsed': parsed_dates.notna(),
}).groupby('label')['date_parsed'].agg(['sum', 'count', 'mean']))

,missing_values,blank_strings,unique_values
column,,,
title,0,0,38729
text,0,631,38646
subject,0,0,8
date,0,0,2397


Dates parsed: 33,285 / 44,898
Unparseable dates: 11613
Date coverage: 2015-05-01 to 2017-12-31
Date parsing by label:


,sum,count,mean
label,,,
fake,11868,23481,0.50543
real,21417,21417,1.00000


## 5. Audit interpretation and modeling guardrails

### Findings from the executed audit

- The combined data contains **44,898 rows** and four candidate input fields (`title`, `text`, `subject`, `date`); with target and provenance columns it uses **152.25 MiB** in this kernel.
- The classes are **23,481 fake (52.3%)** and **21,417 real (47.7%)**. Macro F1 remains the declared primary metric because the two labels have similar support and should contribute equally.
- There are no null values in the four candidate fields. However, **631 `text` values are blank strings**, so “no missing values” does not mean every article has body text.
- Exact duplicates after the first occurrence number **3 in `Fake.csv`** and **206 in `True.csv`**. In total, **405 rows participate in repeated article-field groups**; no repeated article-field group has conflicting labels. Deduplication should be considered together with split design so copies cannot cross train and validation folds.
- Date parsing succeeded for **33,285 of 44,898 rows**. It parsed all **21,417 real** records but only **11,868 fake** records (50.5%); the parseable range is **2015-05-01 to 2017-12-31**. This imbalance makes `date` a strong collection/source proxy candidate, and these parsed dates are not yet a defensible common timeline.
- `subject` has only eight distinct values. Combined with labels originating from separate files, this warrants close inspection for label/source associations. Exclude `source_file` from model inputs.

These findings describe this dataset only. They do not establish publisher coverage or real-world detection performance. Notebook 2 will visualize class/subject structure, text length, blank bodies, duplicates, and date parsing/coverage to make the risks easier to inspect. Any later validation split should account for duplicate overlap and possible source/time structure; preprocessing must be fitted within each training fold.

### Notebook self-critique
The strongest objection to the proposed primary metric is that macro F1 hides the different consequences of false-real and false-fake decisions. It remains a reasonable single selection metric here because the requested task gives the two labels equal status and observed class support is similar; later evaluation must show per-class precision and recall so the error trade-off remains visible. The larger unresolved threat is dataset-source leakage: date parseability already differs sharply by label, and `subject` has only eight values. The visualization and modeling stages must investigate these patterns; this audit alone cannot establish external validity, so claims must remain limited to the sampled dataset.